In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/ames_cleaned.csv', keep_default_na=False, na_values=[''])

In [3]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

Series([], dtype: float64)

In [4]:
# House Age and Remodel Features
df['House_Age'] = df['Yr Sold'] - df['Year Built']
df['Remod_Age'] = df['Yr Sold'] - df['Year Remod/Add']
df['Was_Remodeled'] = (df['Year Remod/Add'] != df['Year Built']).astype(int)

# Total Square Footage
df['Total_SF'] = df['Total Bsmt SF'] + df['1st Flr SF'] + df['2nd Flr SF']
df['Total_Bathrooms'] = (df['Full Bath'] + df['Bsmt Full Bath'] + 
                         0.5 * (df['Half Bath'] + df['Bsmt Half Bath']))

# Porch Indicator
df['Has_Screen_Porch'] = (df['Screen Porch'] > 0).astype(int)
df['Has_Enclosed_Porch'] = (df['Enclosed Porch'] > 0).astype(int)

In [5]:
new_features = ['House_Age', 'Remod_Age', 'Was_Remodeled', 'Total_SF', 'Total_Bathrooms']
print(df[new_features + ['SalePrice']].corr()['SalePrice'].drop('SalePrice'))

House_Age         -0.558921
Remod_Age         -0.534957
Was_Remodeled     -0.047466
Total_SF           0.793127
Total_Bathrooms    0.636175
Name: SalePrice, dtype: float64


In [6]:
df.drop(columns=['Was_Remodeled', 'Pool Area', '3Ssn Porch', 'Low Qual Fin SF', 
                 'Misc Val', 'BsmtFin SF 2', 'PID', 'Order'], inplace=True)
log_features = ['Lot Area', 'Mas Vnr Area', 'Wood Deck SF', 'Open Porch SF',
                '1st Flr SF', 'Lot Frontage', 'Gr Liv Area', 'Total Bsmt SF', 'Total_SF']
for col in log_features:
    df[f'{col}_log'] = np.log1p(df[col])

In [7]:
log_cols = [f'{col}_log' for col in log_features]
print(df[log_cols + ['SalePrice']].corr()['SalePrice'].drop('SalePrice'))

Lot Area_log         0.365084
Mas Vnr Area_log     0.440237
Wood Deck SF_log     0.337949
Open Porch SF_log    0.437554
1st Flr SF_log       0.605155
Lot Frontage_log     0.333034
Gr Liv Area_log      0.695075
Total Bsmt SF_log    0.328320
Total_SF_log         0.764957
Name: SalePrice, dtype: float64


In [8]:
drop_log = ['Gr Liv Area_log', '1st Flr SF_log', 'Total Bsmt SF_log', 'Total_SF_log']

df.drop(columns=drop_log, inplace=True)

In [9]:
remaining_originals = ['Mas Vnr Area', 'Open Porch SF', 'Lot Area', 'Wood Deck SF', 'Lot Frontage']
print(df[remaining_originals + ['SalePrice']].corr()['SalePrice'].drop('SalePrice'))

Mas Vnr Area     0.502196
Open Porch SF    0.312951
Lot Area         0.266549
Wood Deck SF     0.327143
Lot Frontage     0.350473
Name: SalePrice, dtype: float64


In [10]:
# Keep log versions, drop originals
df.drop(columns=['Open Porch SF', 'Lot Area'], inplace=True)

# Keep originals, drop log versions
df.drop(columns=['Mas Vnr Area_log', 'Lot Frontage_log', 'Wood Deck SF_log'], inplace=True)

In [11]:
df.shape

(2930, 81)

In [12]:
neighborhood_medians = df.groupby('Neighborhood')['SalePrice'].median()

tier_map = pd.qcut(neighborhood_medians, q=3, labels=[3, 2, 1]).to_dict()

df['Neighborhood_Tier'] = df['Neighborhood'].map(tier_map)

print(df['Neighborhood_Tier'].isna().sum())
print(df.groupby('Neighborhood_Tier')['SalePrice'].median())

0
Neighborhood_Tier
1    230000.0
2    161000.0
3    124000.0
Name: SalePrice, dtype: float64


In [13]:
df.drop(columns=['Neighborhood'], inplace=True)

In [14]:
qual_map = {'No Basement': 0, 'No Garage': 0, 'No Pool': 0, 'No Fireplace': 0,
            'Unknown': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

qual_cols = ['Bsmt Qual', 'Bsmt Cond', 'Fireplace Qu', 
             'Garage Qual', 'Garage Cond', 'Pool QC', 'Exter Qual', 'Exter Cond',
             'Kitchen Qual', 'Heating QC']

for col in qual_cols:
    df[col] = df[col].map(qual_map)

df['Bsmt Exposure'] = df['Bsmt Exposure'].map(
    {'No Basement': 0, 'Unknown': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
)

bsmt_fin_map = {'No Basement': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
df['BsmtFin Type 1'] = df['BsmtFin Type 1'].map(bsmt_fin_map)

df['Garage Finish'] = df['Garage Finish'].map(
    {'No Garage': 0, 'Unknown':0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
)

df['Fence'] = df['Fence'].map(
    {'No Fence': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}
)

df['Lot Shape'] = df['Lot Shape'].map(
    {'IR3': 1, 'IR2': 2, 'IR1': 3, 'Reg': 4}
)

df['Land Slope'] = df['Land Slope'].map(
    {'Sev': 1, 'Mod': 2, 'Gtl': 3}
)

df['Utilities'] = df['Utilities'].map(
    {'ELO': 1, 'NoSeWa': 2, 'NoSewr': 3, 'AllPub': 4}
)

df['Paved Drive'] = df['Paved Drive'].map(
    {'N': 0, 'P': 1, 'Y': 2}
)

df['Functional'] = df['Functional'].map(
    {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8}
)

In [15]:
print(df[qual_cols + ['Bsmt Exposure', 'BsmtFin Type 1', 'Garage Finish', 
                       'Fence', 'Lot Shape', 'Land Slope', 'Paved Drive', 
                       'Functional']].isna().sum())

Bsmt Qual         0
Bsmt Cond         0
Fireplace Qu      0
Garage Qual       0
Garage Cond       0
Pool QC           0
Exter Qual        0
Exter Cond        0
Kitchen Qual      0
Heating QC        0
Bsmt Exposure     0
BsmtFin Type 1    0
Garage Finish     0
Fence             0
Lot Shape         0
Land Slope        0
Paved Drive       0
Functional        0
dtype: int64


In [16]:
cat_cols = df.select_dtypes(exclude='number').columns
print(cat_cols)
print(f'\n{len(cat_cols)} categorical columns remaining')

for col in cat_cols:
    print(f'\n{col}: {df[col].nunique()} unique values')
    print(df[col].value_counts())

Index(['MS Zoning', 'Street', 'Alley', 'Land Contour', 'Lot Config',
       'Condition 1', 'Condition 2', 'Bldg Type', 'House Style', 'Roof Style',
       'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type',
       'Foundation', 'BsmtFin Type 2', 'Heating', 'Central Air', 'Electrical',
       'Garage Type', 'Misc Feature', 'Sale Type', 'Sale Condition'],
      dtype='str')

23 categorical columns remaining

MS Zoning: 7 unique values
MS Zoning
RL         2273
RM          462
FV          139
RH           27
C (all)      25
I (all)       2
A (agr)       2
Name: count, dtype: int64

Street: 2 unique values
Street
Pave    2918
Grvl      12
Name: count, dtype: int64

Alley: 3 unique values
Alley
No Alley Access    2732
Grvl                120
Pave                 78
Name: count, dtype: int64

Land Contour: 4 unique values
Land Contour
Lvl    2633
HLS     120
Bnk     117
Low      60
Name: count, dtype: int64

Lot Config: 5 unique values
Lot Config
Inside     2140
Corner      511
CulD

In [17]:
# Binary Encoding
df['Street'] = (df['Street'] == 'Pave').astype(int)
df['Central Air'] = (df['Central Air'] == 'Y').astype(int)
df['Is_GasA'] = (df['Heating'] == 'GasA').astype(int)
df['Is_CompShg'] = (df['Roof Matl'] == 'CompShg').astype(int)
df['Is_Normal_Cond1'] = (df['Condition 1'] == 'Norm').astype(int)
df['Is_Normal_Cond2'] = (df['Condition 2'] == 'Norm').astype(int)
df['Has_Misc_Feature'] = (df['Misc Feature'] != 'None').astype(int)

df.drop(columns=['Heating', 'Roof Matl', 'Condition 1', 'Condition 2', 'Misc Feature'], inplace=True)

# Collapse infrequent classes
# Roof Style
df['Roof Style'] = df['Roof Style'].replace(['Gambrel', 'Flat', 'Mansard', 'Shed'], 'Other')

# House Style
df['House Style'] = df['House Style'].replace(['2.5Unf', '1.5Unf', '2.5Fin'], 'Other')

# MS Zoning
df['MS Zoning'] = df['MS Zoning'].replace(['I (all)', 'A (agr)'], 'Other')

# Electrical
df['Electrical'] = df['Electrical'].replace(['FuseP', 'Mix'], 'Other')

# Lot Config
df['Lot Config'] = df['Lot Config'].replace('FR3', 'FR2')

# Mas Vnr Type
df['Mas Vnr Type'] = df['Mas Vnr Type'].replace(['CBlock', 'Unknown'], 'Other')

# BsmtFin Type 2
df['BsmtFin Type 2'] = df['BsmtFin Type 2'].replace('Unknown', 'Unf')

# Exterior 1st — keep top 8, collapse rest into Other
top_ext1 = df['Exterior 1st'].value_counts().head(8).index
df['Exterior 1st'] = df['Exterior 1st'].where(df['Exterior 1st'].isin(top_ext1), 'Other')

# Exterior 2nd — same approach
top_ext2 = df['Exterior 2nd'].value_counts().head(8).index
df['Exterior 2nd'] = df['Exterior 2nd'].where(df['Exterior 2nd'].isin(top_ext2), 'Other')

# Sale Type — keep top 4, collapse rest
top_sale = df['Sale Type'].value_counts().head(4).index
df['Sale Type'] = df['Sale Type'].where(df['Sale Type'].isin(top_sale), 'Other')

# One hot encoding
ohe_cols = ['MS Zoning', 'Alley', 'Land Contour', 'Lot Config', 'Bldg Type',
            'House Style', 'Roof Style', 'Exterior 1st', 'Exterior 2nd',
            'Mas Vnr Type', 'Foundation', 'BsmtFin Type 2', 'Electrical',
            'Garage Type', 'Sale Type', 'Sale Condition']

df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

In [18]:
print(df.shape)
print(df.dtypes.value_counts())

(2930, 138)
bool       73
int64      47
float64    18
Name: count, dtype: int64


In [19]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
print(df.dtypes.value_counts())

int64      120
float64     18
Name: count, dtype: int64


In [20]:
df.to_csv('../data/ames_featured.csv', index=False)